# SPAM: Scalable Polynomial Additive Model

SPAM represents homogeneous polynomial blocks with low-rank projections, avoiding explicit construction of every polynomial coefficient tensor.


## Model


For degree $q$ and rank $R_q$,

$$
p_q(x)=\sum_{r=1}^{R_q}\alpha_{qr}(w_{qr}^{\top}x)^q,
\qquad
\eta(x)=\beta_0+u(x)+\sum_{q=2}^{Q}p_q(x),
$$

where $u(x)$ contains unary effects and diagonal corrections.


## Shared estimator API

All neural estimators use `fit`, `predict`, `score`, `evaluate`, and
`predict_components`. The component result reconstructs predictions on the link
scale and supports shared term-importance and plotting utilities. Constructor
options such as `numerical_preprocessing` and `categorical_preprocessing` are forwarded to
PreTab and are fitted on training rows only.


In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

rng = np.random.default_rng(7)
n = 180
X = pd.DataFrame({
    "x1": rng.uniform(-1.0, 1.0, n),
    "x2": rng.normal(size=n),
    "group": rng.choice(["a", "b", "c"], size=n),
})
y = (
    np.sin(np.pi * X["x1"])
    + 0.35 * X["x2"] ** 2
    + 0.30 * (X["group"] == "b")
    + rng.normal(0.0, 0.12, n)
)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=7
)

# Set True to run the small fit and all fitted-model demonstrations.
RUN_TRAINING = False


## Construct the estimator


In [ ]:
from nampy.models import SPAMClassifier, SPAMLSS, SPAMRegressor


model = SPAMRegressor(
    ranks=[16, 8],       # ranks for degrees 2 and 3
    reg_order=2,
    regularization_scale=1e-5,
    basis_l1_regularization=0.0,
    use_geometric_mean=True,
)
model.get_params(deep=False)


## Fit and inspect

Enable `RUN_TRAINING` above for a short demonstration. Real work should use a
larger validation set, enough epochs, and early stopping.


In [ ]:
if RUN_TRAINING:
    model.fit(
        X_train,
        y_train,
        max_epochs=3,
        batch_size=64,
        random_state=7,
        logger=False,
        enable_progress_bar=False,
        enable_model_summary=False,
    )
    predictions = model.predict(X_test)
    r2 = model.score(X_test, y_test)
    metrics = model.evaluate(X_test, y_test)
    components = model.predict_components(X_test, center=True)
    components.validate_additive_reconstruction()
    display({"R2": r2, **metrics})
    display(model.term_importance(X_test).head())


## Model-specific controls

`local_term_importance` expands the low-rank representation into sample-specific unary and distinct-variable polynomial terms.


In [ ]:
if RUN_TRAINING:
    local = model.local_term_importance(X_test.iloc[:5], top_k=5)
    display(local[0])
    display(model.term_importance(X_test))


## Task variants and limits

`SPAMRegressor`, `SPAMClassifier`, and `SPAMLSS` share the same polynomial basis and regularization controls.
